In [1]:
from pathlib import Path
# 1. Imports

def get_api_key():
    """Retrieve the OpenAI API key from a hidden file."""
    home_dir = Path.home()
    key_file = home_dir / ".secrets" / "openai_api_key.txt"

    if not key_file.exists():
        raise FileNotFoundError(
            f"API Key file not found at {key_file}. "
            "Please ensure the key is saved correctly."
        )

    with key_file.open("r") as f:
        api_key = f.read().strip()

    if not api_key:
        raise ValueError("API Key file is empty. Provide a valid API key.")

    return api_key

from langchain.agents import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver

# 2. Define a simple tool
@tool
def magic_function(input: int) -> int:
    """Example tool: Adds 2 to the input."""
    return input + 2

tools = [magic_function]

# 3. Initialize LLM and Agent
llm = ChatOpenAI(model="gpt-4o", temperature=0, api_key= get_api_key())

react_agent = create_react_agent(
    llm,
    tools=tools,
    prompt="You are a thoughtful agent that can use a magic function tool."
)

graph = StateGraph(react_agent.config.schema)
graph.add_node("agent", react_agent)
graph.set_entry_point("agent")
# loop until agent decides to finish
graph.add_edge("agent", "agent")

# Optional: Add memory checkpointer
memory = MemorySaver()
graph = graph.compile(checkpointer=memory)

agent_executor = graph

# 4. Run the agent
def run_agent(query: str, thread_id: str = "test1"):
    state = {"messages": [HumanMessage(content=query)]}
    config = {"configurable": {"thread_id": thread_id}}
    result = agent_executor.invoke(state, config)
    # Final AI response
    ai_msg = next(msg for msg in result["messages"] if msg.__class__.__name__ != "ToolMessage")
    print("AI:", ai_msg.content)

# Example
run_agent("What is magic_function(3)?")


/usr/local/lib/python3.11/dist-packages/langchain/chains/api/base.py:57: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from langchain_community.utilities.requests import TextRequestsWrapper


AttributeError: 'str' object has no attribute 'input_variables'